In [5]:
import sys
sys.path.insert(0, "/Users/bohaoli/Desktop/tuto/tuto_langchain/official/langchain-langgraph-langsmith/utils")

from utils import ppm, ppms, debug

# Assistants 

[Assistants](https://docs.langchain.com/langsmith/assistants) give developers a quick and easy way to modify and version agents for experimentation.

## Supplying configuration to the graph

Our `task_maistro` graph is already set up to use assistants!

It has a `configuration.py` file defined and loaded in the graph.

We access configurable fields (`user_id`, `todo_category`, `task_maistro_role`) inside the graph nodes.

## Creating assistants 

Now, what is a practical use case for assistants with the `task_maistro` app that we've been building?

For me, it's the ability to have separate ToDo lists for different categories of tasks. 

For example, I want one assistant for my personal tasks and another for my work tasks.

These are easily configurable using the `todo_category` and `task_maistro_role` configurable fields.

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

This is the default assistant that we created when we deployed the graph.

### 🔍 Config 到底在什么时候被使用？

你传入的 `config={"configurable": {"todo_category": "personal"}}` **不是在创建 assistant 时执行的**，而是：

1. 创建 assistant 时，config 被**存到数据库**（Postgres）
2. 当你用这个 assistant 执行 run 时，LangGraph API 自动把存储的 config **注入到每个 node** 的 `RunnableConfig` 参数中
3. Node 函数内部调用 `Configuration.from_runnable_config(config)` **提取值**

具体在 `task_maistro.py` 的 node 里：

```python
def some_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    # 这里读取 config
    configurable = configuration.Configuration.from_runnable_config(config)
    user_id = configurable.user_id           # "lance"
    todo_category = configurable.todo_category  # "personal" 或 "work"
    
    # 用 todo_category 构建 memory store 的 namespace
    namespace = (user_id, todo_category)
    memories = store.search(namespace)
    ...
```

所以 `todo_category: "personal"` 决定了这个 assistant 读写 memory 时用哪个 namespace：
- Personal assistant → 只看到 personal 任务
- Work assistant → 只看到 work 任务

**Config 不是在创建时「执行」的，而是在每次 run 的每个 node 执行时被读取。**

In [6]:
from langgraph_sdk import get_client

url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### Personal assistant

This is the personal assistant that I'll use to manage my personal tasks.

In [7]:
personal_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": {"todo_category": "personal"}}
)
debug(personal_assistant)

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "c799d79e-1ddf-4b08-8541-3f66a4b2211e",                                                       │
│   "graph_id": "task_maistro",                                                                                   │
│   "version": 1,                                                                                                 │
│   "created_at": "2026-05-09T02:51:29.432381+00:00",                                                             │
│   "updated_at": "2026-05-09T02:51:29.432381+00:00",                                                             │
│   "config": {                                                                                                   │
│     "configurable": {                                                                                           │
│       "todo_category": "personal"                                                                               │
│     }                                                                                                           │
│   },                                                                                                            │
│   "context": {                                                                                                  │
│     "todo_category": "personal"                                                                                 │
│   },                                                                                                            │
│   "metadata": {},                                                                                               │
│   "name": "Untitled",                                                                                           │
│   "description": null                                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Let's update this assistant to include my `user_id` for convenience,  [creating a new version of it](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant). 

In [10]:
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

configurations = {
    "todo_category": "personal",
    "user_id": "lance",
    "task_maistro_role": task_maistro_role,
}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"], config={"configurable": configurations}
)
debug(personal_assistant)

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "c799d79e-1ddf-4b08-8541-3f66a4b2211e",                                                       │
│   "graph_id": "task_maistro",                                                                                   │
│   "version": 4,                                                                                                 │
│   "created_at": "2026-05-09T02:51:29.432381+00:00",                                                             │
│   "updated_at": "2026-05-09T02:52:41.766000+00:00",                                                             │
│   "config": {                                                                                                   │
│     "configurable": {                                                                                           │
│       "task_maistro_role": "You are a friendly and organized personal task assistant. Your main focus is helpin │
│       "todo_category": "personal",                                                                              │
│       "user_id": "lance"                                                                                        │
│     }                                                                                                           │
│   },                                                                                                            │
│   "context": {                                                                                                  │
│     "task_maistro_role": "You are a friendly and organized personal task assistant. Your main focus is helping  │
│     "todo_category": "personal",                                                                                │
│     "user_id": "lance"                                                                                          │
│   },                                                                                                            │
│   "metadata": {},                                                                                               │
│   "name": "Untitled",                                                                                           │
│   "description": null                                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Work assistant

Now, let's create a work assistant. I'll use this for my work tasks.

In [11]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {
    "todo_category": "work",
    "user_id": "lance",
    "task_maistro_role": task_maistro_role,
}

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro",
    config={"configurable": configurations},
)
print(work_assistant)

{'assistant_id': 'b4e256dc-f469-4ec6-ad5c-1be8fa12e70f', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-05-09T02:53:09.242307+00:00', 'updated_at': '2026-05-09T02:53:09.242307+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks base

## Using assistants 

Assistants will be saved to `Postgres` in our deployment.  

This allows us to easily search <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [search](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) for assistants with the SDK.

In [12]:
assistants = await client.assistants.search()
for assistant in assistants:
    debug(
        {
            "assistant_id": assistant["assistant_id"],
            "version": assistant["version"],
            "config": assistant["config"],
        }
    )

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "b4e256dc-f469-4ec6-ad5c-1be8fa12e70f",                                                       │
│   "version": 1,                                                                                                 │
│   "config": {                                                                                                   │
│     "configurable": {                                                                                           │
│       "task_maistro_role": "You are a focused and efficient work task assistant. \n\nYour main focus is helping │
│       "todo_category": "work",                                                                                  │
│       "user_id": "lance"                                                                                        │
│     }                                                                                                           │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "c799d79e-1ddf-4b08-8541-3f66a4b2211e",                                                       │
│   "version": 4,                                                                                                 │
│   "config": {                                                                                                   │
│     "configurable": {                                                                                           │
│       "task_maistro_role": "You are a friendly and organized personal task assistant. Your main focus is helpin │
│       "todo_category": "personal",                                                                              │
│       "user_id": "lance"                                                                                        │
│     }                                                                                                           │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "c426e627-217b-4a1e-9724-50cbbc8a4edb",                                                       │
│   "version": 1,                                                                                                 │
│   "config": {                                                                                                   │
│     "configurable": {                                                                                           │
│       "todo_category": "personal"                                                                               │
│     }                                                                                                           │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "0794c2bd-c22b-497f-a04a-2b409401e83a",                                                       │
│   "version": 1,                                                                                                 │
│   "config": {                                                                                                   │
│     "configurable": {                                                                                           │
│       "todo_category": "personal"                                                                               │
│     }                                                                                                           │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧠 Debug ────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "assistant_id": "ea4ebafa-a81d-5063-a5fa-67c755d98a21",                                                       │
│   "version": 1,                                                                                                 │
│   "config": {}                                                                                                  │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

We can manage them easily with the SDK. For example, we can delete assistants that we're no longer using.  
> The syntax in the video is slightly off. The updated code below creates a spare assistant and then deletes it. 

In [13]:
# create a temporary assitant
temp_assistant = await client.assistants.create(
    "task_maistro", config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")

# delete our temporary assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': 6f3087ce-a197-4020-b1cd-fe31654581a9}
before delete: {'assistant_id': b4e256dc-f469-4ec6-ad5c-1be8fa12e70f}
before delete: {'assistant_id': c799d79e-1ddf-4b08-8541-3f66a4b2211e}
before delete: {'assistant_id': c426e627-217b-4a1e-9724-50cbbc8a4edb}
before delete: {'assistant_id': 0794c2bd-c22b-497f-a04a-2b409401e83a}
before delete: {'assistant_id': ea4ebafa-a81d-5063-a5fa-67c755d98a21}

after delete: {'assistant_id': 6f3087ce-a197-4020-b1cd-fe31654581a9 }
after delete: {'assistant_id': b4e256dc-f469-4ec6-ad5c-1be8fa12e70f }
after delete: {'assistant_id': c799d79e-1ddf-4b08-8541-3f66a4b2211e }
after delete: {'assistant_id': c426e627-217b-4a1e-9724-50cbbc8a4edb }
after delete: {'assistant_id': 0794c2bd-c22b-497f-a04a-2b409401e83a }


Let's set the assistant IDs for the `personal` and `work` assistants that I'll work with.

In [14]:
work_assistant_id = assistants[0]["assistant_id"]
personal_assistant_id = assistants[1]["assistant_id"]

### Work assistant

Let's add some ToDos for my work assistant.

In [15]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday."
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    work_assistant_id,
    input={"messages": [HumanMessage(content=user_input)]},
    stream_mode="values",
):
    if chunk.event == "values":
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_oAozb7BvkME2VUgzYoGboJhK)
 Call ID: call_oAozb7BvkME2VUgzYoGboJhK
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Re-film Module 6, lesson 5', 'time_to_complete': 120, 'deadline': '2026-05-09T23:59:59', 'solutions': ['Book a quiet room', 'Prepare the script', 'Check camera and audio equipment'], 'status': 'not started'}
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_iZ6qMoLOpDiloO7fOKQDHdbT)
 Call ID: call_iZ6qMoLOpDiloO7fOKQDHdbT
  Args:
    update_type: todo
================================= Tool Message ===========

In [16]:
user_input = "Create another ToDo: Finalize set of report generation tutorials."
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    work_assistant_id,
    input={"messages": [HumanMessage(content=user_input)]},
    stream_mode="values",
):
    if chunk.event == "values":
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create another ToDo: Finalize set of report generation tutorials.
================================== Ai Message ==================================

Could you provide a deadline for the task "Finalize set of report generation tutorials"? Based on similar tasks, this might take around 2 days. Would you like to set a deadline with this in mind?


The assistant uses it's instructions to push back with task creation! 

It asks me to specify a deadline :) 

In [17]:
user_input = "OK, for this task let's get it done by next Tuesday."
async for chunk in client.runs.stream(
    thread["thread_id"],
    work_assistant_id,
    input={"messages": [HumanMessage(content=user_input)]},
    stream_mode="values",
):
    if chunk.event == "values":
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

OK, for this task let's get it done by next Tuesday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_bepJSTSoAAeEjTg427g89Url)
 Call ID: call_bepJSTSoAAeEjTg427g89Url
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Finalize set of report generation tutorials', 'time_to_complete': 2880, 'deadline': '2026-05-12T23:59:59', 'solutions': ['Outline the tutorial structure', 'Create video content', 'Review and edit videos', 'Upload and publish tutorials'], 'status': 'not started'}
================================== Ai Message ==================================

I've updated the task "Finalize set of report generation tutorials" with a deadline of next Tuesday, May 12, 2026. If there's anything else you'd like to add or adjust, feel free to let me kn

### Personal assistant

Similarly, we can add ToDos for my personal assistant.

In [18]:
user_input = "Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points."
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    personal_assistant_id,
    input={"messages": [HumanMessage(content=user_input)]},
    stream_mode="values",
):
    if chunk.event == "values":
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_hbvIFp8l8iYmZXx2uPsLx7ti)
 Call ID: call_hbvIFp8l8iYmZXx2uPsLx7ti
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Check on swim lessons for the baby this weekend', 'time_to_complete': 30, 'deadline': '2026-05-14T23:59:59', 'solutions': ['Call the local swimming pool', 'Check online for available classes', 'Confirm the schedule and book a slot'], 'status': 'not started'}

New ToDo created:
Content: {'task': 'For winter travel, check AmEx points', 'time_to_complete': 45, 'solutions': ['Log into AmEx account', 'Check points balance', 'Review travel redemption options'], 'status': 'not started

In [ ]:
user_input = "Give me a todo summary."
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    personal_assistant_id,
    input={"messages": [HumanMessage(content=user_input)]},
    # stream_mode="values",
):
    if chunk.event == "values":
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Give me a todo summary.
================================== Ai Message ==================================

Here's your current ToDo summary:

**Overdue:**
- None

**Today:**
- None

**This Week:**
- Re-film Module 6, lesson 5 (Deadline: 2026-05-09)

**Future:**
- Update audioUX (Deadline: 2026-05-11)
- Finalize set of report generation tutorials (Deadline: 2026-05-12)
- Check on swim lessons for the baby this weekend (Deadline: 2026-05-14)

**Tasks Missing Deadlines:**
- For winter travel, check AmEx points: I notice this task doesn't have a deadline yet. Based on similar tasks, this might take about 45 minutes. Would you like to set a deadline with this in mind?

**Tasks Lacking Time Estimates:**
- Update audioUX: This task seems important but lacks a time estimate. Would you like to add one?

Let me know if you'd like to make any updates or set deadlines!
